# RQ3: Link Prediction Reliability Evaluation

This notebook evaluates the quality of predicted paper-to-paper links generated by the multi-agent ingestion pipeline against the **Semantic Scholar Citation Graph** as the sole gold standard.

### Research Question & Hypothesis:
- **RQ3:** *“How accurately can LLM agents identify and establish the correct directional links between documents?”*
- **H3:** *“The Link Agent will identify a meaningful proportion of the directed citation links between papers in the corpus, although some errors are expected because the agent predicts links from paper summaries rather than complete reference lists.”*

### Evaluations Performed:
1. **Primary Result: Strict Directed Metrics** (Precision, Recall, F1 where direction matters)
2. **Secondary Analysis: Undirected Pair Metrics** (Precision, Recall, F1 ignoring direction)

## Semantic Scholar API Data Structure Example

When querying references for a paper (e.g., `arXiv:2007.01282`) using the Semantic Scholar API endpoint:
`https://api.semanticscholar.org/graph/v1/paper/arXiv:{arxiv_id}/references?fields=citedPaper.externalIds`

The raw JSON response from the API has the following format:
```json
{
  "data": [
    {
      "citedPaper": {
        "paperId": "df3c467a...",
        "externalIds": {
          "ArXiv": "2005.11401",
          "DOI": "10.48550/arXiv.2005.11401"
        }
      }
    }
  ]
}
```

Our preprocessing script (`scripts/build_citation_gold.py`) extracts these `externalIds.ArXiv` values and builds the clean gold citation graph file (`data/gold_labels/citations_gold.json`), which we load below.

In [1]:
import json
import re
from pathlib import Path
import pandas as pd

# Define paths
notes_dir = Path("data/generated_notes")
s2_links_path = Path("data/gold_labels/citations_gold.json")
output_path = Path("data/results/link_eval.json")

In [2]:
# Print an example of the parsed and stored gold citation links
with open(s2_links_path, encoding="utf-8") as f:
    s2_links = json.load(f)

print(f"Loaded {len(s2_links)} gold citations.")
print("Sample of resolved citation link records in citations_gold.json:")
print(json.dumps(s2_links[:3], indent=2))

Loaded 42 gold citations.
Sample of resolved citation link records in citations_gold.json:
[
  {
    "source": "2007.01282",
    "target": "2005.11401",
    "link_type": "cites"
  },
  {
    "source": "2012.13635",
    "target": "1711.11157",
    "link_type": "cites"
  },
  {
    "source": "2212.12393",
    "target": "2012.13635",
    "link_type": "cites"
  }
]


In [3]:
# Resolve paper titles to arXiv IDs
title_to_arxiv = {}
for note_path in notes_dir.glob("*.md"):
    arxiv_id = note_path.stem
    try:
        content = note_path.read_text(encoding="utf-8")
    except Exception:
        continue
    title_match = re.search(r'^title:\s*"(.*?)"', content, re.MULTILINE)
    if not title_match:
        title_match = re.search(r'^title:\s*(.*?)$', content, re.MULTILINE)
    if title_match:
        title = title_match.group(1).strip().strip('"').strip("'")
        title_to_arxiv[title.lower()] = arxiv_id

print(f"Resolved {len(title_to_arxiv)} corpus paper titles.")

Resolved 40 corpus paper titles.


In [4]:
# 2. Parse predicted links from notes
predicted_links = []

for note_path in notes_dir.glob("*.md"):
    source_id = note_path.stem  # arXiv ID
    try:
        content = note_path.read_text(encoding="utf-8")
    except Exception as e:
        print(f"Error reading {note_path}: {e}")
        continue
        
    parts = content.split("## Generated Links")
    if len(parts) < 2:
        continue
        
    links_section = parts[1]
    lines = links_section.strip().split("\n")
    
    for line in lines:
        line = line.strip()
        if not line.startswith("-"):
            continue
            
        match = re.search(r"-\s*\*\*\[\[(.*?)\]\]\*\*\s*\((.*?)\)", line)
        if match:
            target_title = match.group(1).strip()
            target_id = title_to_arxiv.get(target_title.lower())
            
            if target_id:
                predicted_links.append({
                    "source": source_id,
                    "target": target_id
                })

print(f"Total parsed predicted links: {len(predicted_links)}")

Total parsed predicted links: 189


In [5]:
# 3. Set up directed and undirected sets
corpus_arxiv_ids = {p.stem for p in notes_dir.glob("*.md")}

pred_edges_dir = {(l["source"], l["target"]) for l in predicted_links}
gold_edges_dir = {
    (l["source"], l["target"]) for l in s2_links 
    if l["source"] in corpus_arxiv_ids and l["target"] in corpus_arxiv_ids
}

pred_edges_undir = {frozenset([u, v]) for u, v in pred_edges_dir}
gold_edges_undir = {frozenset([u, v]) for u, v in gold_edges_dir}

print(f"Directed: Predicted={len(pred_edges_dir)}, Gold={len(gold_edges_dir)}")
print(f"Undirected: Predicted={len(pred_edges_undir)}, Gold={len(gold_edges_undir)}")

Directed: Predicted=189, Gold=42
Undirected: Predicted=133, Gold=42


In [6]:
# 4. Calculate strict directed metrics (Primary)
tp_dir = len(pred_edges_dir & gold_edges_dir)
fp_dir = len(pred_edges_dir - gold_edges_dir)
fn_dir = len(gold_edges_dir - pred_edges_dir)

p_dir = tp_dir / len(pred_edges_dir) if pred_edges_dir else 0.0
r_dir = tp_dir / len(gold_edges_dir) if gold_edges_dir else 0.0
f1_dir = 2 * p_dir * r_dir / (p_dir + r_dir) if p_dir + r_dir > 0 else 0.0

print("Strict Directed Metrics (Primary):")
print(f"  TP={tp_dir}, FP={fp_dir}, FN={fn_dir}")
print(f"  Precision={p_dir:.5f}, Recall={r_dir:.5f}, F1={f1_dir:.5f}")

Strict Directed Metrics (Primary):
  TP=28, FP=161, FN=14
  Precision=0.14815, Recall=0.66667, F1=0.24242


In [7]:
# 5. Calculate undirected paper metrics (Secondary)
tp_undir = len(pred_edges_undir & gold_edges_undir)
fp_undir = len(pred_edges_undir - gold_edges_undir)
fn_undir = len(gold_edges_undir - pred_edges_undir)

p_undir = tp_undir / len(pred_edges_undir) if pred_edges_undir else 0.0
r_undir = tp_undir / len(gold_edges_undir) if gold_edges_undir else 0.0
f1_undir = 2 * p_undir * r_undir / (p_undir + r_undir) if p_undir + r_undir > 0 else 0.0

print("Undirected Pair Metrics (Secondary):")
print(f"  TP={tp_undir}, FP={fp_undir}, FN={fn_undir}")
print(f"  Precision={p_undir:.5f}, Recall={r_undir:.5f}, F1={f1_undir:.5f}")

Undirected Pair Metrics (Secondary):
  TP=37, FP=96, FN=5
  Precision=0.27820, Recall=0.88095, F1=0.42286


In [8]:
# 6. Format results as a pandas DataFrame
df_data = {
    "Metric": [
        "True Positives (TP)",
        "False Positives (FP)",
        "False Negatives (FN)",
        "Precision",
        "Recall",
        "F1-Score"
    ],
    "Strict Directed (Primary)": [
        tp_dir,
        fp_dir,
        fn_dir,
        f"{p_dir:.5f}",
        f"{r_dir:.5f}",
        f"{f1_dir:.5f}"
    ],
    "Undirected Pair (Secondary)": [
        tp_undir,
        fp_undir,
        fn_undir,
        f"{p_undir:.5f}",
        f"{r_undir:.5f}",
        f"{f1_undir:.5f}"
    ]
}

results_df = pd.DataFrame(df_data)
results_df

,Metric,Strict Directed (Primary),Undirected Pair (Secondary)
0,True Positives (TP),28,37
1,False Positives (FP),161,96
2,False Negatives (FN),14,5
3,Precision,0.14815,0.27820
4,Recall,0.66667,0.88095
5,F1-Score,0.24242,0.42286


In [9]:
# 7. Save final structured JSON
results = {
    "evaluation_scope": "RQ3 - Link Prediction Reliability (Sole Gold Standard: Semantic Scholar API)",
    "hypothesis": "H3: The Link Agent will identify a meaningful proportion of the directed citation links between papers in the corpus, although some errors are expected because the agent predicts links from paper summaries rather than complete reference lists.",
    "directed_evaluation_primary": {
        "true_positives": tp_dir,
        "false_positives": fp_dir,
        "false_negatives": fn_dir,
        "precision": round(p_dir, 5),
        "recall": round(r_dir, 5),
        "f1": round(f1_dir, 5)
    },
    "undirected_evaluation_secondary": {
        "true_positives": tp_undir,
        "false_positives": fp_undir,
        "false_negatives": fn_undir,
        "precision": round(p_undir, 5),
        "recall": round(r_undir, 5),
        "f1": round(f1_undir, 5)
    }
}

output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {output_path}")

Results saved to data\results\link_eval.json
